In [1]:
import sys
from pathlib import Path

# Add src/ to sys.path regardless of where Jupyter is launched from
for _candidate in [Path().resolve().parent / "src", Path().resolve() / "src"]:
    if _candidate.exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))
        print(f"Added to sys.path: {_candidate}")
        break

Added to sys.path: C:\Users\loren\Documents\Postdoc\Compressed_sensing\compressed_sensing_bioacoustics\src


In [2]:
from preprocess import Preprocessing
from settings import Config
from config_species import get_settings
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pickle

In [ ]:
selected_species = "thyolo" #"thyolo" # gibbon / ptw
settings = get_settings(selected_species)
config = Config(settings) # Pass settings to your config object
print("Loaded settings for:", selected_species)
print(config.preprocessing.dict())
print(config.data.dict())

In [ ]:
config.data.species_folder=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Thyolo"

config.preprocessing.audio_extension=".npy"

method_compression="cs"
parameter_compression="0.2"

In [ ]:
preprocess = Preprocessing(
            **config.preprocessing.dict(),
            species_folder=config.data.species_folder,
            positive_class=config.data.positive_class,
            negative_class=config.data.negative_class,
        )

In [ ]:
#parameter_list=["8k","40k", "96k"]
#for parameter_compression in parameter_list : 
    
types=[ "train", "val", "test"]

for dataset_type in types:
    print(dataset_type)
    if dataset_type=="train":
        X_calls, Y_calls = preprocess.create_dataset(dataset_type, method_compression=method_compression, parameter_compression=parameter_compression, preprocessing=True, data_augmentation=True, noise_reduction=False)
    else:
        X_calls, Y_calls = preprocess.create_dataset(dataset_type, method_compression=method_compression, parameter_compression=parameter_compression, preprocessing=True, data_augmentation=False,  noise_reduction=False)
    Y = preprocess._one_hot_encode(Y_calls)
    
    #save data into pickle files
    saving_path=Path(preprocess.species_folder, dataset_type)       
    if method_compression != None : 
        with open(Path(saving_path, str(config.data.positive_class) + "_X_" + dataset_type + "_" + method_compression + "_"+ parameter_compression+ ".pkl"), "wb") as f:
                                pickle.dump(X_calls, f) 
    else : 
        with open(Path(saving_path, str(config.data.positive_class) + "_X_" + dataset_type + ".pkl"), "wb") as f:
                                pickle.dump(X_calls, f) 

        with open(Path(saving_path, str(config.data.positive_class) + "_Y_" + dataset_type + ".pkl"), "wb") as f:
                                pickle.dump(Y, f)   

## Plot the obtained dataset

In [ ]:
def load_dataset(species_folder, species, dataset_type="train", method_compression=None, parameter_compression=None) -> tuple:
        """Loads the dataset from the save path"""
        # Load X dataset         
        if method_compression!=None:
            with open(Path(species_folder, dataset_type, species + "_X_" + dataset_type + "_" + method_compression + "_"+ parameter_compression + ".pkl"), "rb") as f:
                X = pickle.load(f)
            print("Dataset loaded from " + str(species_folder) + f"/{dataset_type}_{method_compression}_{parameter_compression}")
        else : 
            with open(Path(species_folder, dataset_type, species + "_X_" + dataset_type + ".pkl"), "rb") as f:
                X = pickle.load(f)
            print("Dataset loaded from " + str(species_folder) + f"/{dataset_type}_baseline")
        
        #load Y
        with open(Path(species_folder, dataset_type, species + "_Y_" + dataset_type + ".pkl"), "rb") as f:
                Y = pickle.load(f)   
        
        return X, Y

In [ ]:
X_train, Y_train= load_dataset(config.data.species_folder, config.data.positive_class, dataset_type="train", method_compression=method_compression, parameter_compression=parameter_compression)
X_test, Y_test= load_dataset(config.data.species_folder, config.data.positive_class, dataset_type="test", method_compression=method_compression, parameter_compression=parameter_compression)
X_val, Y_val= load_dataset(config.data.species_folder, config.data.positive_class, dataset_type="val", method_compression=method_compression, parameter_compression=parameter_compression)
Y=Y_val.copy()
X=X_val.copy()

In [ ]:
print("number segments training dataset : ", len(X_train), len(Y_train))
print("number segments validation dataset : ", len(X_val),len(Y_val))
print("number segments testing dataset : ", len(X_test),len(Y_test))

In [ ]:
index_positive=np.where(np.argmax(Y, axis=1)==1)[0]
index_negative=np.where(np.argmax(Y, axis=1)==0)[0]

In [ ]:
index_p=np.random.choice(index_positive, size=12, replace=False)
index_n=np.random.choice(index_negative, size=12, replace=False)

In [ ]:
index_p=np.random.choice(index_positive, size=12, replace=False)

fig = plt.figure(figsize=(10,10))
w=0
for i in index_p:
    plt.subplot(4,3,w+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X[i],  origin='lower')
    plt.xlabel(Y[i], fontsize=13)
    w+=1
plt.tight_layout()



In [ ]:

index_n=np.random.choice(index_negative, size=12, replace=False)
fig = plt.figure(figsize=(10,10))
w=0
for i in index_n:
    plt.subplot(4,3,w+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X[i],  origin='lower')
    plt.xlabel(Y[i], fontsize=13)
    w+=1
plt.tight_layout()